In [ ]:
# ============================================================
# CUDA/NVRTC Error Fix - Run this FIRST
# ============================================================
# This cell fixes the "libnvrtc-builtins.so.12.6" error
# by disabling JIT compilation and setting up the environment

import os
import sys

# Disable PyTorch JIT compilation globally
os.environ['PYTORCH_JIT'] = '0'
os.environ['TORCH_COMPILE_DISABLE'] = '1'

# Try to install CUDA runtime if missing (for Kaggle)
try:
    import subprocess
    # Check if nvrtc libraries exist
    result = subprocess.run(['find', '/usr/local/cuda*', '-name', 'libnvrtc*.so*', '2>/dev/null'], 
                          shell=True, capture_output=True, text=True)
    if not result.stdout.strip():
        print("⚠ CUDA NVRTC libraries not found in standard locations")
        print("  Attempting to work around by disabling JIT compilation...")
except:
    pass

print("✓ Environment configured to avoid NVRTC errors")


In [ ]:
import sys
import subprocess
from pathlib import Path
import torch

# ============================================================
# STEP 1: Install packages
# ============================================================

DATASET_PATH = "/kaggle/input/rfdetr-forgery-detection/Ruthra_packages/rfdetr_offline_packages"

print("="*60)
print("Installing packages from offline source...")
print("="*60)

# Install fairscale from nested directory
fairscale_inner = Path(DATASET_PATH) / "fairscale-0.4.13" / "fairscale-0.4.13"

if fairscale_inner.exists():
    print(f"\n✓ Found fairscale at: {fairscale_inner}")
    print("Installing fairscale...")
    
    result = subprocess.run([
        sys.executable, "-m", "pip", "install",
        str(fairscale_inner),
        "--no-deps",
        "-q"
    ], capture_output=True, text=True)
    
    if result.returncode == 0:
        print("✓ fairscale installed successfully!")
    else:
        print("Error:", result.stderr)
else:
    print(f"✗ fairscale not found at: {fairscale_inner}")

# Install other packages (rfdetr, supervision, etc.)
print("\n" + "="*60)
print("Installing other packages...")
print("="*60)

packages_dir = Path(DATASET_PATH)

# Find and install all packages
for item in packages_dir.iterdir():
    if item.name == "fairscale-0.4.13":
        continue  # Already handled
    
    # Check for directories with setup.py or pyproject.toml
    if item.is_dir():
        if (item / "setup.py").exists() or (item / "pyproject.toml").exists():
            print(f"\nInstalling {item.name}...")
            try:
                subprocess.run([
                    sys.executable, "-m", "pip", "install",
                    str(item),
                    "--no-deps",
                    "-q"
                ], check=True, capture_output=True)
                print(f"  ✓ {item.name} installed")
            except subprocess.CalledProcessError:
                print(f"  ✗ {item.name} failed")
    
    # Install wheel files
    elif item.suffix == ".whl":
        print(f"\nInstalling {item.name}...")
        try:
            subprocess.run([
                sys.executable, "-m", "pip", "install",
                str(item),
                "--no-deps",
                "-q"
            ], check=True, capture_output=True)
            print(f"  ✓ {item.name} installed")
        except subprocess.CalledProcessError:
            print(f"  ✗ {item.name} failed")

# ============================================================
# STEP 2: Verify imports
# ============================================================

print("\n" + "="*60)
print("Verifying imports...")
print("="*60)

try:
    import fairscale
    print("✓ fairscale")
except ImportError as e:
    print(f"✗ fairscale: {e}")

try:
    import supervision as sv
    print("✓ supervision")
except ImportError as e:
    print(f"✗ supervision: {e}")

try:
    import rfdetr
    from rfdetr import RFDETRSegPreview
    print("✓ rfdetr")
    print("✓ RFDETRSegPreview")
except ImportError as e:
    print(f"✗ rfdetr: {e}")
    raise

# ============================================================
# STEP 3: Load your trained model
# ============================================================

# print("\n" + "="*60)
# print("Loading RF-DETR model...")
# print("="*60)

# # Path to your model weights
# MODEL_WEIGHTS_PATH = "/kaggle/input/rfdetr-forgery-detection/checkpoint0017.pth"  # Update this path

# # Initialize the model
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# print(f"Using device: {device}")

# # Create model instance
# model = RFDETRSegPreview(
#     num_classes=1,  # Update based on your training (1 for binary, or number of forgery classes)
#     pretrain_weights=MODEL_WEIGHTS_PATH  # We're loading custom weights
# )

# print('Model loaded')
# # Load your trained weights
# print(f"\nLoading weights from: {MODEL_WEIGHTS_PATH}")
# checkpoint = torch.load(MODEL_WEIGHTS_PATH, map_location=device)

# # Load state dict (handle different checkpoint formats)
# if isinstance(checkpoint, dict):
#     if 'model' in checkpoint:
#         model.load_state_dict(checkpoint['model'])
#         print("✓ Loaded from checkpoint['model']")
#     elif 'state_dict' in checkpoint:
#         model.load_state_dict(checkpoint['state_dict'])
#         print("✓ Loaded from checkpoint['state_dict']")
#     else:
#         model.load_state_dict(checkpoint)
#         print("✓ Loaded state dict directly")
# else:
#     model.load_state_dict(checkpoint)
#     print("✓ Loaded state dict directly")

# # Move model to device and set to eval mode
# model = model.to(device)
# model.eval()

# print("✓ Model loaded successfully!")

# # ============================================================
# # STEP 4: Prepare for inference
# # ============================================================

# print("\n" + "="*60)
# print("Model ready for inference!")
# print("="*60)

# # Example inference function
# def predict_forgery(image_path, confidence_threshold=0.5):
#     """
#     Predict forgeries in an image
    
#     Args:
#         image_path: Path to the image file
#         confidence_threshold: Minimum confidence score for detections
    
#     Returns:
#         detections: Detection results
#     """
#     import cv2
#     import numpy as np
#     from torchvision import transforms
    
#     # Load image
#     image = cv2.imread(image_path)
#     image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
#     # Prepare image for model
#     transform = transforms.Compose([
#         transforms.ToTensor(),
#         transforms.Normalize(mean=[0.485, 0.456, 0.406], 
#                            std=[0.229, 0.224, 0.225])
#     ])
    
#     image_tensor = transform(image_rgb).unsqueeze(0).to(device)
    
#     # Run inference
#     with torch.no_grad():
#         outputs = model(image_tensor)
    
#     # Process outputs (adjust based on your model's output format)
#     # This is a generic example - modify based on RF-DETR's actual output
#     boxes = outputs['boxes'][0].cpu().numpy()
#     scores = outputs['scores'][0].cpu().numpy()
#     labels = outputs['labels'][0].cpu().numpy()
    
#     # Filter by confidence threshold
#     mask = scores > confidence_threshold
#     boxes = boxes[mask]
#     scores = scores[mask]
#     labels = labels[mask]
    
#     return {
#         'boxes': boxes,
#         'scores': scores,
#         'labels': labels,
#         'image': image
#     }

# print("\nYou can now use the model for predictions:")
# print("  results = predict_forgery('path/to/image.jpg')")
# print("\nModel is ready! 🚀")

Installing packages from offline source...

✓ Found fairscale at: /kaggle/input/rfdetr-forgery-detection/Ruthra_packages/rfdetr_offline_packages/fairscale-0.4.13/fairscale-0.4.13
Installing fairscale...
Error:   error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× pip subprocess to install build dependencies did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


Installing other packages...

Installing torchvision-0.24.1-cp310-cp310-manylinux_2_28_x86_64.whl...
  ✗ torchvision-0.24.1-cp310-cp310-manylinux_2_28_x86_64.whl failed

Installing requests_toolbelt-1.0.0-py2.py3-none-any.whl...
  ✓ requests_toolbelt-1.0.0-py2.py3-none-any.whl installed

Installing 

2026-01-12 16:08:08.969941: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768234089.148066      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768234089.197267      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768234089.626138      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768234089.626194      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768234089.626197      55 computation_placer.cc:177] computation placer alr

✓ rfdetr
✓ RFDETRSegPreview


In [ ]:
# Fix CUDA JIT compilation error by disabling JIT
import os
import torch

# Disable PyTorch JIT compilation to avoid NVRTC errors
os.environ['PYTORCH_JIT'] = '0'
torch.jit._state.disable()

# Also try to disable CUDA JIT if possible
try:
    torch.backends.cudnn.enabled = False  # Sometimes helps
except:
    pass

# Set device - try CPU if CUDA has issues
if torch.cuda.is_available():
    try:
        # Test if CUDA works
        test_tensor = torch.zeros(1).cuda()
        device = 'cuda'
        print(f"✓ CUDA available, using GPU")
    except Exception as e:
        print(f"⚠ CUDA available but has issues: {e}")
        print("  Falling back to CPU")
        device = 'cpu'
        os.environ['CUDA_VISIBLE_DEVICES'] = ''  # Hide CUDA
else:
    device = 'cpu'
    print(f"✓ Using CPU")

# Initialize model
print("Initializing RFDETRSegPreview model...")
model = RFDETRSegPreview(pretrain_weights='/kaggle/input/rfdetr-forgery-detection/checkpoint0017.pth')

# Move to device
model = model.to(device)
model.eval()
print("✓ Model loaded and ready")

Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
Using patch size 12 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
Loading pretrain weights


In [ ]:
from PIL import Image
import numpy as np
import torch

# Disable JIT before prediction to avoid NVRTC errors
torch.jit._state.disable()

# Load image
img_path = '/kaggle/input/recodai-luc-scientific-image-forgery-detection/train_images/authentic/10070.png'
img = Image.open(img_path)

# Convert to RGB (handles grayscale, RGBA, and other formats)
if img.mode != 'RGB':
    img = img.convert('RGB')

# Now predict with error handling
try:
    print("Running prediction...")
    with torch.no_grad():
        # Disable JIT compilation during inference
        torch._C._set_graph_executor_optimize(False)
        prediction = model.predict(img)
    print("✓ Prediction successful!")
    print(f"Prediction type: {type(prediction)}")
except RuntimeError as e:
    if "nvrtc" in str(e).lower() or "libnvrtc" in str(e).lower():
        print("✗ CUDA JIT compilation error detected")
        print("\nTrying workaround: Force CPU mode...")
        
        # Force CPU mode
        model_cpu = model.cpu()
        torch._C._set_graph_executor_optimize(False)
        
        try:
            with torch.no_grad():
                prediction = model_cpu.predict(img)
            print("✓ Prediction successful on CPU!")
        except Exception as e2:
            print(f"✗ Still failed: {e2}")
            print("\nSOLUTION: You may need to:")
            print("1. Install correct CUDA libraries")
            print("2. Use a different PyTorch version")
            print("3. Modify rfdetr to disable JIT compilation")
            raise
    else:
        raise

RuntimeError: 
  #ifdef __HIPCC__
  #define ERROR_UNSUPPORTED_CAST ;
  // corresponds to aten/src/ATen/native/cuda/thread_constants.h
  #define CUDA_OR_ROCM_NUM_THREADS 256
  // corresponds to aten/src/ATen/cuda/detail/OffsetCalculator.cuh
  #define MAX_DIMS 16
  #ifndef __forceinline__
  #define __forceinline__ inline __attribute__((always_inline))
  #endif
  #else
  //TODO use _assert_fail, because assert is disabled in non-debug builds
  #define ERROR_UNSUPPORTED_CAST assert(false);
  #define CUDA_OR_ROCM_NUM_THREADS 128
  #define MAX_DIMS 25
  #endif
  #define POS_INFINITY __int_as_float(0x7f800000)
  #define INFINITY POS_INFINITY
  #define NEG_INFINITY __int_as_float(0xff800000)
  #define NAN __int_as_float(0x7fffffff)

  typedef long long int int64_t;
  typedef unsigned int uint32_t;
  typedef signed char int8_t;
  typedef unsigned char uint8_t;  // NOTE: this MUST be "unsigned char"! "char" is equivalent to "signed char"
  typedef short int16_t;
  static_assert(sizeof(int64_t) =